# 05 · 전체 흐름 스모크 테스트

**00만 실행한 뒤에도** 실행할 수 있습니다. 실제 원본 파일을 읽거나 기존 학습 작업을 시작하지 않습니다.
합성 거래 CSV 133행(중복 제거 후 132건), 계좌 CSV, 패턴 TXT를 생성하여
전처리 → 일별 피처 → 그래프 → 전체 인덱스 → 검증 → 4개 모델 각 1에폭 → 전체 Val/Test를 실행합니다.
결과는 WORK_DIR 아래의 별도 `smoke_runs/` 폴더에 저장합니다.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

PROJECT_DIR = Path.cwd().resolve()
link = PROJECT_DIR / ".salt_project.json"
if not link.exists():
    raise FileNotFoundError("이 노트북들이 있는 폴더에서 00_환경설정과_실행준비.ipynb를 먼저 실행하세요.")
project = json.loads(link.read_text())
WORK_DIR = Path(project["work_dir"])
DEVICE = project["device"]
for module_name in ("salt_daily", "runtime", "run", "notebook_control"):
    loaded = sys.modules.get(module_name)
    location = getattr(loaded, "__file__", None)
    if location and not Path(location).resolve().is_relative_to(WORK_DIR):
        raise RuntimeError("다른 WORK_DIR가 import되어 있습니다. 커널을 재시작하세요.")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

def command(stage, device=DEVICE):
    subprocess.run([sys.executable, "-u", str(WORK_DIR/"run.py"), stage, "--device", device], cwd=WORK_DIR, check=True)

print("작업 폴더:", WORK_DIR)


### 이 단계의 실행 코드

아래 소스는 00 노트북이 읽어 작업 폴더에 실행 파일로 만듭니다. 코드를 수정하면 노트북을 저장하고 **새 WORK_DIR**로 00을 다시 실행하세요.

In [ ]:
SOURCES = {}


In [ ]:
# salt_daily/raw_example.py
SOURCES['salt_daily/raw_example.py'] = '''"""Synthetic original-format CSVs covering each fixed experimental window."""
import csv
from pathlib import Path
from datetime import datetime,timedelta
from .raw_common import HEADERS,ACCOUNT_HEADERS


def make_example(folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    rows=[];known=[]
    for day in (0,6,7,8,55,56,64,69,73,74,91):
        for i in range(12):
            s=i%8;d=(i*3+day)%8
            stamp=datetime(2022,8,1)+timedelta(days=day,minutes=i//3)
            amount=str(10+i*7+day)
            row=[stamp.strftime('%Y/%m/%d %H:%M'),str(s%2),f'{s:06X}',str(d%2),f'{d:06X}',
                 amount,'US Dollar',amount,'US Dollar',('ACH','Wire','Cash')[i%3],str(int(i in (0,7)))]
            rows.append(row)
            if i==0:known.append(row)
    rows.append(rows[0].copy())
    with (folder/'HI-Large_Trans.csv').open('w',newline='') as f:
        writer=csv.writer(f);writer.writerow(HEADERS);writer.writerows(rows)
    with (folder/'HI-Large_accounts.csv').open('w',newline='') as f:
        writer=csv.writer(f);writer.writerow(ACCOUNT_HEADERS)
        for i in range(9):writer.writerow([f'USA Bank #{i%2}',str(i%2),f'{i:06X}',f'entity-{i//2}',f'Individual #{i//2}'])
    with (folder/'HI-Large_Patterns.txt').open('w',newline='') as f:
        f.write('BEGIN LAUNDERING ATTEMPT - BIPARTITE\\n');csv.writer(f).writerows(known)
        f.write('END LAUNDERING ATTEMPT - BIPARTITE\\n')
    return dict(transactions=str(folder/'HI-Large_Trans.csv'),accounts=str(folder/'HI-Large_accounts.csv'),patterns=str(folder/'HI-Large_Patterns.txt'))
'''


In [ ]:
# salt_daily/smoke.py
SOURCES['salt_daily/smoke.py'] = '''"""Run the complete staged pipeline on tiny synthetic original-format files."""
import argparse
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from .config import ROOT
from .raw_example import make_example
from .storage import read_json,write_json


def run_smoke(parent=None, device='cpu'):
    if device not in ('cpu', 'cuda'):
        raise ValueError(device)
    parent=Path(parent) if parent else ROOT/'smoke_runs'
    parent.mkdir(parents=True,exist_ok=True)
    target=Path(tempfile.mkdtemp(prefix='raw_to_test_',dir=parent))
    for name in ('salt_daily','runtime','tests','docs'):
        shutil.copytree(ROOT/name,target/name,ignore=shutil.ignore_patterns('__pycache__','*.so'))
    for name in ('run.py','notebook_control.py','protocol.template.json'):
        shutil.copy2(ROOT/name,target/name)
    files=make_example(target/'raw')
    write_json(target/'local_settings.json',dict(data_root=str(target/'data')))
    write_json(target/'raw_settings.json',dict(files=files,preprocess=dict(require_hi_large_counts=False,memory_limit='512MB',threads=1)))
    protocol=read_json(target/'protocol.template.json')
    protocol['training'].update(epochs_primary=1,learning_rates_primary=[.0012],logical_batch=16,microbatch_max=8)
    # Tiny fixture only: avoid taking a production-sized CUDA allocator budget.
    if device == 'cuda':
        protocol['training']['cuda_memory_fraction'] = .03
    protocol['graph'].update(sampler_backend=device, target_batch_size=8)
    protocol['input_pipeline']['backend'] = device
    protocol['cohorts']={}
    write_json(target/'protocol.template.json',protocol)
    for stage in ('ingest','features','nodes','graph','prepare','validate','run'):
        with (target/f'{stage}.log').open('w') as output:
            result=subprocess.run([sys.executable,'-u',str(target/'run.py'),stage,'--device',device],cwd=target,
                                  stdout=output,stderr=subprocess.STDOUT)
        if result.returncode:
            raise RuntimeError(f'{stage} failed; see {target/stage}.log\\n'+(target/f'{stage}.log').read_text()[-5000:])
        print(stage,'PASS',flush=True)
    final=read_json(target/'runs/daily_close_001/complete.json')
    assert set(final['test'])=={'A3','A0','A1','A2'}
    assert final['test']['A3']['test_authorization']['scope']=='variant'
    events=[json.loads(line) for line in (target/'runs/daily_close_001/events.jsonl').read_text().splitlines()]
    assert next(i for i,e in enumerate(events) if e['event']=='variant_final_evaluation_started' and e['variant']=='A3') < next(i for i,e in enumerate(events) if e['event']=='variant_started' and e['variant']=='A0')
    for item in final['test'].values():assert item['scored']==24 and item['coverage']==1.
    result=dict(status='pass',synthetic_only=True,device=device,raw_rows=133,retained_rows=132,models=4,
                epochs=1,full_test_scored_per_model=24,output=str(target))
    write_json(target/'smoke_result.json',result)
    print(json.dumps(result,ensure_ascii=False),flush=True)
    return result


if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--output',type=Path)
    parser.add_argument('--device',choices=('cpu','cuda'),default='cpu')
    args=parser.parse_args()
    run_smoke(args.output,args.device)
'''


In [ ]:
# tests/test_raw_preprocessing.py
SOURCES['tests/test_raw_preprocessing.py'] = '''"""Independent checks of raw-window features, deduplication and daily snapshots."""
import tempfile
import unittest
from pathlib import Path
import numpy as np
import duckdb
from salt_daily.raw_common import ingest,build_transaction_features,build_node_snapshot
from salt_daily.raw_config import RawConfig
from salt_daily.raw_example import make_example


class RawPreprocessingTests(unittest.TestCase):
    def setUp(self):
        self.temp=tempfile.TemporaryDirectory();self.root=Path(self.temp.name)
        files=make_example(self.root/'raw')
        self.c=duckdb.connect();self.c.execute('SET threads=1')
        self.cfg=RawConfig(require_hi_large_counts=False)
        ingest(self.c,files['transactions'],files['accounts'],self.cfg)
        build_transaction_features(self.c,self.cfg)

    def tearDown(self):
        self.c.close();self.temp.cleanup()

    def test_observed_dedup_and_minute_features_against_explicit_history(self):
        rows=self.c.execute('SELECT edge_id,src,dst,ts FROM observed ORDER BY edge_id').fetchall()
        self.assertEqual(len(rows),132)
        self.assertNotIn('y',[r[0] for r in self.c.execute('DESCRIBE observed').fetchall()])
        features=dict((r[0],r[1:]) for r in self.c.execute('SELECT edge_id,f0,f1,f2,f3,f4,f5,f6 FROM features').fetchall())
        for eid,src,dst,stamp in rows:
            histories=[[t for _,s,d,t in rows if t<stamp and condition(s,d)] for condition in
                       (lambda s,d:s==src and d==dst,lambda s,d:s==src,lambda s,d:d==dst)]
            expected=[]
            for j,history in enumerate(histories):
                if j==0:expected.append(np.log1p(len(history)))
                expected.append(np.log(2+(stamp-max(history)).total_seconds()/60) if history else 0.)
                expected.append(np.log1p(sum((stamp-t).total_seconds()<=3600 for t in history)))
            np.testing.assert_allclose(features[eid],expected,atol=1e-10)

    def test_labels_do_not_enter_features_and_daily_snapshot_stops_at_cutoff(self):
        before=self.c.execute('SELECT * FROM features ORDER BY edge_id').fetchall()
        self.c.execute('UPDATE tx SET y=1-y')
        for name in ('features','calibration','scales','robust','windows','pair_days','history','ports','first_pair'):
            self.c.execute('DROP TABLE '+name)
        build_transaction_features(self.c,self.cfg)
        self.assertEqual(before,self.c.execute('SELECT * FROM features ORDER BY edge_id').fetchall())
        nodes=self.c.execute('SELECT count(*) FROM nodes').fetchone()[0]
        build_node_snapshot(self.c,self.root,'day0','2022-08-02',nodes,10)
        x=np.load(self.root/'nodes_day0.npy')
        expected=self.c.execute("SELECT src,count(*) FROM observed WHERE day=DATE '2022-08-01' GROUP BY src").fetchall()
        for node,count in expected:self.assertAlmostEqual(x[node,0],np.log1p(count),places=6)
        self.assertEqual(x.shape,(nodes,19));self.assertTrue(np.isfinite(x).all())
'''


In [ ]:
# tests/test_extra_features.py
SOURCES['tests/test_extra_features.py'] = '''import unittest
import numpy as np
import torch

from salt_daily.extra_features import ExtraEdgeFeatures, encode_formats
from salt_daily.model import DailySALT


class ExtraFeatureTests(unittest.TestCase):
    def test_unknown_payment_codes_and_target_order(self):
        encoder = ExtraEdgeFeatures.__new__(ExtraEdgeFeatures)
        encoder.formats = np.array([0, 6, 7, -1, 255])
        encoder.in_ports = np.array([0, 3, 2, 1, 4])
        encoder.out_ports = np.array([2, 1, 0, 3, 4])
        ids = np.array([4, 0, 3, 0, 1])
        result = encoder.append(np.zeros((len(ids), 14), np.float32), ids)
        self.assertEqual(result.shape, (5, 24))
        np.testing.assert_array_equal(result[:, 14:22].sum(1), np.ones(5))
        np.testing.assert_array_equal(result[:, 14:22].argmax(1), [7, 0, 7, 0, 6])
        np.testing.assert_array_equal(result[:, 22], encoder.in_ports[ids])
        np.testing.assert_array_equal(result[:, 23], encoder.out_ports[ids])
        np.testing.assert_array_equal(encode_formats(np.arange(8)), np.eye(8))

    def test_new_features_reach_message_and_target_encoders_all_variants(self):
        torch.manual_seed(42)
        for variant in ('A0', 'A1', 'A2', 'A3'):
            model = DailySALT(variant, hidden=4, edge_dim=24, dropout=0, final_dropout=0)
            x = torch.randn(3, 20)
            index = torch.tensor([[0, 1, 2, 0], [1, 2, 0, 2]])
            attributes = torch.randn(4, 24, requires_grad=True)
            target_features = torch.randn(2, 24, requires_grad=True)
            flags = torch.ones(4, dtype=torch.bool)
            logits = model(x, index, attributes, index[:, :2], target_features,
                           flags, flags, torch.ones(3, 2), return_logits=True)
            logits.square().sum().backward()
            self.assertEqual(logits.shape, (2, 2))
            self.assertTrue(torch.isfinite(logits).all())
            self.assertGreater(float(attributes.grad[:, 14:].abs().sum()), 0)
            self.assertGreater(float(target_features.grad[:, 14:].abs().sum()), 0)
'''


In [ ]:
# tests/test_full_population.py
SOURCES['tests/test_full_population.py'] = '''import copy
import json
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch

import numpy as np
import torch
import torch.nn.functional as F

from salt_daily.population import build_population, FullPopulation
from salt_daily.metrics import confusion, ranking, summarize
from salt_daily.evaluation import ScoreStore
from salt_daily.training import microbatch_loss, loss_denominator
from salt_daily.storage import write_json

torch.set_num_threads(1)


def dataset(root):
    days = np.array([7, 7, 7, 8, 8, 56, 56, 56, 56, 69, 69, 69, 69, 74, 74, 74, 74])
    labels = np.array([0, 1, 0, 9, 0, 0, 1, 0, 9, 0, 1, 0, 9, 0, 1, 0, 9], np.int8)
    positive = np.flatnonzero(labels)
    build_population(root/'cache/population', days*1440, np.ones(len(days), np.uint8),
                     positive, labels[positive],
                     windows={'train_gnn': (7, 8), 'monitor': (56, 56),
                              'validation': (69, 69), 'test': (74, 74)}, block=3)
    protocol = {'variants': ['A3'], 'architecture': {'common_hidden': 4, 'heads': 2,
                'dropout': .083, 'final_dropout': .288, 'edge_dim': 24},
                'training': {'seed': 42, 'epochs_primary': 2, 'logical_batch': 2,
                             'positive_weight': 7, 'learning_rates_primary': [.0012]*2,
                             'microbatch_max': 1, 'max_edges_per_microbatch': 40000, 'gradient_clip': 5},
                'evaluation': {'score_commit_targets': 2}}
    write_json(root/'protocol.json', protocol)
    write_json(root/'code_manifest.json', {})
    write_json(root/'calibration.json', {'delta': [1., 1.]})
    return protocol


class TinyLoader:
    def set_day(self, day):
        self.day = day

    def microbatches(self, data, indices, microbatch=1, max_edges=40000):
        # One independent graph per target. Inputs depend on ID, never its label.
        for index in indices:
            tid = int(data['ids'][index])
            rng = np.random.default_rng(tid)
            x = rng.normal(size=(3, 20)).astype('float32')
            edge = np.array([[0, 0, 1, 2], [1, 1, 2, 0]], dtype='int64')
            attributes = rng.normal(size=(4, 24)).astype('float32')
            target = np.array([[0], [1]], dtype='int64')
            flags = np.ones(4, dtype=bool)
            degree = np.stack((np.bincount(edge[1], minlength=3), np.bincount(edge[0], minlength=3)), axis=1).astype('float32')
            arrays = (x, edge, attributes, target, attributes[:1], flags, flags, degree)
            yield np.array([index]), arrays, dict(nodes=3, edges=4)


class PopulationTests(unittest.TestCase):
    def test_all_targets_once_no_label_selection_and_no_weights(self):
        with tempfile.TemporaryDirectory() as temp:
            root = Path(temp)
            minute = np.array([8, 7, 9, 7, 8, 6, 9])*1440
            present = np.array([1, 1, 1, 0, 1, 1, 1], np.uint8)
            first = build_population(root/'one', minute, present, np.array([1, 6]), np.array([1, 9]),
                                     windows={'train_gnn': (7, 8), 'test': (9, 9)}, block=2)
            build_population(root/'two', minute, present, np.array([0, 4]), np.array([9, 1]),
                             windows={'train_gnn': (7, 8), 'test': (9, 9)}, block=3)
            for scope in first['cohorts']:
                a = FullPopulation(scope, root/'one'); b = FullPopulation(scope, root/'two')
                np.testing.assert_array_equal(a.ids, b.ids)
                with self.assertRaises(KeyError):
                    a['w']
            data = FullPopulation('train_gnn', root/'one')
            np.testing.assert_array_equal(data.ids, [1, 0, 4])
            orders = list(data.orders(3))
            np.testing.assert_array_equal(np.sort(np.concatenate([o for _, o in orders])), np.arange(3))
            np.testing.assert_array_equal(list(data.orders(3, start_day_cursor=1))[0][1], orders[1][1])
            self.assertEqual(data.updates(2), 2)

    def test_weighted_ce_microbatch_loss_and_gradients_match_torch(self):
        torch.manual_seed(123)
        full = torch.randn(7, 2, dtype=torch.float64, requires_grad=True)
        part = full.detach().clone().requires_grad_()
        labels = torch.tensor([0, 1, 0, 0, 1, 0, 1])
        expected = F.cross_entropy(full, labels, weight=torch.tensor([1., 7.], dtype=torch.float64))
        expected.backward()
        denominator = loss_denominator(labels.numpy())
        actual = sum(microbatch_loss(part[s:e], labels[s:e], denominator)
                     for s, e in [(0, 1), (1, 4), (4, 7)])
        actual.backward()
        torch.testing.assert_close(actual, expected, rtol=1e-14, atol=1e-14)
        torch.testing.assert_close(full.grad, part.grad, rtol=1e-14, atol=1e-14)
        self.assertEqual(denominator, 25.)

    def test_integer_metrics_all_targets_ties_and_thresholds(self):
        y = np.array([1, 0, 9, 0]); scores = np.array([2., 1., 1., -1.]); ids = np.array([8, 9, 3, 2])
        result = summarize(y, scores, ids)
        self.assertEqual((result['TP'], result['FP'], result['FN'], result['TN']), (2, 1, 0, 1))
        self.assertAlmostEqual(result['precision'], 2/3)
        self.assertAlmostEqual(result['AP'], 5/6)
        self.assertEqual(result['threshold_metrics'][-1]['TP'], 0)
        ap, top = ranking(y, scores, ids, fractions=(.5,), block=2)
        self.assertAlmostEqual(ap, 5/6)
        self.assertEqual(top[0]['TP'], 2)  # ID 3 wins the score tie over ID 9.
        self.assertFalse(result['population_ipw'])
        with self.assertRaises(ValueError):
            confusion(y, [2., float('nan'), 1., -1.])

    def test_ap_ties_cross_multiple_chunks_against_independent_reference(self):
        rng = np.random.default_rng(7)
        y = rng.integers(0, 2, size=103)
        scores = rng.integers(-3, 4, size=103).astype('float32')
        expected = 0.; previous = 0
        for threshold in sorted(set(scores), reverse=True):
            pred = scores >= threshold; tp = int(y[pred].sum())
            expected += (tp-previous)/y.sum()*tp/pred.sum(); previous = tp
        for block in (1, 2, 7, 1000):
            ap, _ = ranking(y, scores, np.arange(len(y)), block=block)
            self.assertAlmostEqual(ap, expected, places=14)

    def test_score_commits_resume_and_reject_corruption_or_changed_model(self):
        with tempfile.TemporaryDirectory() as temp:
            folder = Path(temp)
            request = dict(rows=5, model='a')
            store = ScoreStore(folder, request); store.commit([1., 2.], 1.)
            store.values[2:] = 999; store.values.flush()  # Uncommitted work must be overwritten.
            resumed = ScoreStore(folder, request)
            self.assertEqual(resumed.cursor, 2)
            resumed.commit([3., 4., 5.], 2.)
            final = ScoreStore(folder, request)
            np.testing.assert_array_equal(final.values, [1, 2, 3, 4, 5])
            with self.assertRaises(ValueError):
                ScoreStore(folder, dict(rows=5, model='b'))
            final.values[0] = -99; final.values.flush()
            with self.assertRaises(AssertionError):
                ScoreStore(folder, request)

    def test_training_resume_matches_full_run_and_full_test_is_frozen(self):
        from salt_daily import training, evaluation
        def execute(root, interrupt=False):
            root.mkdir(exist_ok=True)
            protocol = dataset(root)
            calls = []
            original_guard = training.resource_guard
            def guard(*args):
                calls.append(1)
                if interrupt and len(calls) == 2:
                    raise RuntimeError('test interruption')
            with patch.object(training, 'ROOT', root), patch.object(training, 'RUN', root/'run'), \\
                 patch.object(evaluation, 'ROOT', root), \\
                 patch.object(training, 'cohort', lambda scope: FullPopulation(scope, root/'cache/population')), \\
                 patch.object(training, 'resource_guard', guard), patch.object(training, 'log'), patch.object(evaluation, 'log'):
                training.train_variant('A3', protocol, TinyLoader(), 'cpu')
                validation = training.evaluate_scope(protocol, TinyLoader(), 'validation', 'cpu')
                selection = training.freeze_selection(protocol, validation)
                result = training.evaluate_scope(protocol, TinyLoader(), 'test', 'cpu')
                self.assertEqual(result['A3']['scored'], 4)
                self.assertEqual(result['A3']['coverage'], 1.)
                self.assertFalse(result['A3']['precision_is_population_estimate'])
                self.assertEqual(selection['selected_variant'], 'A3')
                return torch.load(root/'run/models/A3/last.pt', weights_only=False), result
        with tempfile.TemporaryDirectory() as temp:
            root = Path(temp)
            full, full_result = execute(root/'full')
            with self.assertRaisesRegex(RuntimeError, 'test interruption'):
                execute(root/'resume', interrupt=True)
            resumed, resumed_result = execute(root/'resume')
            for name, value in full['model'].items():
                torch.testing.assert_close(value, resumed['model'][name], rtol=0, atol=0)
            self.assertEqual(resumed['progress']['history'][-1]['train_requested'], 5)
            self.assertEqual(resumed['progress']['history'][-1]['optimizer_updates'], 3)
            for key in ('precision', 'recall', 'F1', 'AP', 'TP', 'FP', 'FN', 'TN'):
                self.assertEqual(full_result['A3'][key], resumed_result['A3'][key])

    def test_larger_batch_resume_preserves_all_targets_and_mixed_update_count(self):
        from salt_daily import training, evaluation
        from salt_daily.storage import sha
        class CountingLoader(TinyLoader):
            def __init__(self):
                self.seen = []
            def microbatches(self, data, indices, **kwargs):
                if self.day in (7, 8):
                    self.seen.extend(map(int, indices))
                yield from super().microbatches(data, indices, **kwargs)
        with tempfile.TemporaryDirectory() as temp:
            root = Path(temp); protocol = dataset(root)
            protocol['training']['logical_batch'] = 1
            write_json(root/'protocol.json', protocol)
            data = FullPopulation('train_gnn', root/'cache/population')
            loader = CountingLoader(); calls = []
            def guard(*args):
                calls.append(1)
                if len(calls) == 2:
                    raise RuntimeError('batch migration interruption')
            with patch.object(training, 'ROOT', root), patch.object(training, 'RUN', root/'run'), \\
                 patch.object(evaluation, 'ROOT', root), \\
                 patch.object(training, 'cohort', lambda scope: FullPopulation(scope, root/'cache/population')), \\
                 patch.object(training, 'log'), patch.object(evaluation, 'log'):
                with patch.object(training, 'resource_guard', guard):
                    with self.assertRaisesRegex(RuntimeError, 'batch migration interruption'):
                        training.train_variant('A3', protocol, loader, 'cpu')
                path = root/'run/models/A3/last.pt'
                saved = torch.load(path, weights_only=False)
                self.assertEqual(saved['progress']['epoch_targets'], 1)
                protocol['training']['logical_batch'] = 3
                write_json(root/'protocol.json', protocol)
                saved['protocol_sha256'] = sha(root/'protocol.json')
                torch.save(saved, path)
                self.assertEqual(training.expected_epoch_updates(data, saved['progress'], 3), 3)
                invalid = dict(saved['progress'], epoch_targets=2)
                with self.assertRaises(AssertionError):
                    training.expected_epoch_updates(data, invalid, 3)
                training.train_variant('A3', protocol, loader, 'cpu')
            final = torch.load(path, weights_only=False)
            expected = np.concatenate([order for _, order in data.orders(1)])
            np.testing.assert_array_equal(loader.seen[:5], expected)
            self.assertEqual(len(loader.seen), 10)
            self.assertEqual([h['optimizer_updates'] for h in final['progress']['history']], [3, 2])
            self.assertTrue(all(h['train_scored'] == 5 for h in final['progress']['history']))

    def test_evaluation_interruption_resumes_without_rescoring_committed_targets(self):
        from salt_daily import evaluation
        from salt_daily.model import DailySALT
        class CountLoader(TinyLoader):
            def __init__(self):
                self.seen = []
            def microbatches(self, data, indices, **kwargs):
                self.seen.extend(data['ids'][indices].tolist())
                yield from super().microbatches(data, indices, **kwargs)
        with tempfile.TemporaryDirectory() as temp:
            root = Path(temp); dataset(root)
            data = FullPopulation('test', root/'cache/population'); loader = CountLoader()
            model = DailySALT('A3', hidden=4, edge_dim=24).eval()
            def interrupted():
                if len(loader.seen) >= 2:
                    raise RuntimeError('evaluation interruption')
            with patch.object(evaluation, 'ROOT', root), patch.object(evaluation, 'log'):
                with self.assertRaisesRegex(RuntimeError, 'evaluation interruption'):
                    evaluation.score(model, data, loader, 'cpu', root/'model', 'test', stop_check=interrupted)
                values, _ = evaluation.score(model, data, loader, 'cpu', root/'model', 'test')
                np.testing.assert_array_equal(loader.seen, data.ids)
                self.assertTrue(np.isfinite(values).all())

    def test_best_checkpoint_uses_full_monitor_f1_and_rejects_old_ipw_resume(self):
        from salt_daily import training
        def scores(model, data, loader, device, folder, scope, epoch, stop_check):
            # Epoch 1 has perfect AP but no alerts; epoch 2 has higher F1.
            return np.array([-3., -1., -4., -2.] if epoch == 1 else [3., 2., -1., 1.]), 0.
        with tempfile.TemporaryDirectory() as temp:
            root = Path(temp); protocol = dataset(root)
            with patch.object(training, 'ROOT', root), patch.object(training, 'RUN', root/'run'), \\
                 patch.object(training, 'cohort', lambda scope: FullPopulation(scope, root/'cache/population')), \\
                 patch.object(training, 'resource_guard'), patch.object(training, 'log'), patch.object(training, 'score', scores):
                training.train_variant('A3', protocol, TinyLoader(), 'cpu')
                folder = root/'run/models/A3'
                history = json.loads((folder/'history.json').read_text())
                self.assertGreater(history[0]['AP'], history[1]['AP'])
                self.assertGreater(history[1]['F1'], history[0]['F1'])
                best = torch.load(folder/'best.pt', weights_only=False)
                self.assertEqual(best['epoch'], 2)
                # Simulate an old IPW checkpoint in the new run: it must fail closed.
                (folder/'complete.json').unlink()
                saved = torch.load(folder/'last.pt', weights_only=False)
                saved['population_ipw'] = True
                torch.save(saved, folder/'last.pt')
                with self.assertRaises(AssertionError):
                    training.train_variant('A3', protocol, TinyLoader(), 'cpu')


if __name__ == '__main__':
    unittest.main()
'''


In [ ]:
# tests/test_shared_topology.py
SOURCES['tests/test_shared_topology.py'] = '''"""Native shared sampler against an independently constructed disjoint union."""
import ctypes as C
import unittest
import numpy as np
from salt_daily import config
from runtime.graph_runtime import Topology, load_reference_native
from salt_daily.shared_topology import load_native, SharedTopology, copy_result


def tiny(lib, cls=Topology):
    t=cls.__new__(cls)
    t.src=np.array([0,0,1,2,3,4,0],dtype=np.int32)
    t.dst=np.array([1,1,2,0,1,4,3],dtype=np.int32)
    t.minute=np.array([0,0,1,2,3,4,1440],dtype=np.int32)
    t.nn=5;t.width=len(t.src);t.cap=1;t.lib=lib
    t.rawptr=[a.ctypes.data_as(C.POINTER(C.c_int32)) for a in (t.src,t.dst,t.minute)]
    t.indices=[];t.indexptr=[]
    for endpoints in (t.dst,t.src):
        for priority in (True,False):
            rows=[];offsets=[0]
            for node in range(t.nn):
                ids=np.flatnonzero(endpoints==node)
                order=ids[::-1] if priority else ids[np.lexsort((ids,t.minute[ids]))]
                rows.extend(order);offsets.append(len(rows))
            a=np.array(offsets,dtype=np.int64);b=np.array(rows,dtype=np.uint32)
            t.indices.extend((a,b))
            t.indexptr.extend((a.ctypes.data_as(C.POINTER(C.c_int64)),b.ctypes.data_as(C.POINTER(C.c_uint32))))
    return t


def reference_union(graph,topology,ids):
    nodes=np.array(sorted(set(map(int,graph[0]))),dtype=np.int64)
    mapping={int(n):i for i,n in enumerate(nodes)};flags={}
    for edge,flag in zip(graph[1],graph[6]):flags[int(edge)]=flags.get(int(edge),0)|int(flag)
    edges=np.array(sorted(flags),dtype=np.int64)
    sources=np.array([mapping[int(topology.src[e])] for e in edges],dtype=np.int64)
    destinations=np.array([mapping[int(topology.dst[e])] for e in edges],dtype=np.int64)
    ts=np.array([mapping[int(topology.src[e])] for e in ids],dtype=np.int64)
    td=np.array([mapping[int(topology.dst[e])] for e in ids],dtype=np.int64)
    ego=np.zeros(len(nodes),dtype=np.int64);ego[ts]=1;ego[td]=1
    return nodes,edges,sources,destinations,ts,td,np.array([flags[int(e)] for e in edges]),ego


class SharedTopologyTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.legacy=tiny(load_reference_native())
        cls.shared=tiny(load_native(), SharedTopology)

    def test_union_preserves_parallel_transactions_directions_and_day_cutoff(self):
        ids=np.array([2,0,5])
        expected=reference_union(self.legacy.batch(ids),self.legacy,ids)
        actual=self.shared.batch(ids)
        for a,b in zip(actual,expected):np.testing.assert_array_equal(a,b)
        self.assertIn(0,actual[1]);self.assertIn(1,actual[1])
        self.assertNotIn(6,actual[1])
        self.assertEqual(len(actual[0]),len(set(actual[0])))
        self.assertEqual(len(actual[1]),len(set(actual[1])))
        self.assertIn(3,actual[6])

    def test_single_query_is_identical_to_previous_graph(self):
        for tid in range(6):
            for a,b in zip(self.legacy.batch([tid]),self.shared.batch([tid])):
                np.testing.assert_array_equal(a,b)

    def test_target_permutation_keeps_context_and_maps_each_target(self):
        first=self.shared.batch([0,2,5]);second=self.shared.batch([5,0,2])
        for index in (0,1,2,3,6,7):np.testing.assert_array_equal(first[index],second[index])
        np.testing.assert_array_equal(second[0][second[4]],self.shared.src[[5,0,2]])
        np.testing.assert_array_equal(second[0][second[5]],self.shared.dst[[5,0,2]])

    def test_mixed_days_are_rejected_even_at_native_boundary(self):
        with self.assertRaises(RuntimeError):self.shared.batch([0,6])

    def test_repeated_states_preserve_target_order_and_distinct_minutes(self):
        # Parallel transactions 0/1 share endpoint-minute states; 2/3/4
        # revisit these accounts at different minutes. Repeating a target
        # must not drop a prediction or change the graph union.
        ids=np.array([0,1,0,2,3,4,5,2])
        expected=reference_union(self.legacy.batch(ids),self.legacy,ids)
        actual=self.shared.batch(ids)
        for a,b in zip(actual,expected):np.testing.assert_array_equal(a,b)
        self.assertEqual(len(actual[4]),len(ids))

    def test_variable_length_copy_does_not_grow_ctypes_pointer_cache(self):
        source=np.arange(5000,dtype=np.int64)
        pointer=source.ctypes.data_as(C.POINTER(C.c_int64))
        class Result:
            size=0
            def result_size(self,handle,index):return self.size
            def result_data(self,handle,index):return pointer
        lib=Result();before=len(C._pointer_type_cache)
        for size in range(2000):
            lib.size=size
            actual=copy_result(lib,None,0)
            np.testing.assert_array_equal(actual,source[:size])
            self.assertTrue(actual.flags.owndata)
        self.assertEqual(len(C._pointer_type_cache),before)

    def test_native_handle_freed_when_copy_fails(self):
        from unittest.mock import Mock,patch
        topology=tiny(load_native(), SharedTopology)
        native=topology.lib
        proxy=Mock(wraps=native);topology.lib=proxy
        with patch('salt_daily.shared_topology.copy_result',side_effect=MemoryError):
            with self.assertRaises(MemoryError):topology.batch([0])
        proxy.free_result.assert_called_once()
'''


In [ ]:
# tests/test_cuda_topology.py
SOURCES['tests/test_cuda_topology.py'] = '''"""CUDA must preserve CPU sampling, including uncapped same-minute edges."""
import ctypes as C
import unittest

import numpy as np
import torch

from salt_daily.shared_topology import SharedTopology, load_native
from salt_daily.cuda_topology import CudaTopology


def fixture(cls):
    topology = cls.__new__(cls)
    rng = np.random.default_rng(801)
    topology.src = rng.integers(0, 9, 1600, dtype=np.int32)
    topology.dst = rng.integers(0, 9, 1600, dtype=np.int32)
    topology.minute = rng.integers(0, 4, 1600, dtype=np.int32)
    # Hundreds of parallel transactions at one minute must all survive cap100.
    topology.src[:350] = 0; topology.dst[:350] = 1; topology.minute[:350] = 2
    topology.minute[-100:] = 1440
    topology.nn = 9; topology.width = 1600; topology.cap = 100
    topology.rawptr = [a.ctypes.data_as(C.POINTER(C.c_int32))
                       for a in (topology.src, topology.dst, topology.minute)]
    topology.indices = []; topology.indexptr = []
    for endpoints in (topology.dst, topology.src):
        for priority in (True, False):
            rows = []; offsets = [0]
            for node in range(topology.nn):
                ids = np.flatnonzero(endpoints == node)
                order = ids[::-1] if priority else ids[np.lexsort((ids, topology.minute[ids]))]
                rows.extend(order); offsets.append(len(rows))
            a = np.array(offsets, dtype=np.int64); b = np.array(rows, dtype=np.uint32)
            topology.indices.extend((a, b))
            topology.indexptr.extend((a.ctypes.data_as(C.POINTER(C.c_int64)),
                                      b.ctypes.data_as(C.POINTER(C.c_uint32))))
    return topology


@unittest.skipUnless(torch.cuda.is_available(), 'CUDA device required')
class CudaTopologyTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.cpu = fixture(SharedTopology); cls.cpu.lib = load_native()
        cls.gpu = fixture(CudaTopology); cls.gpu.attach()

    def test_exact_uncapped_same_minute_parallel_edges_and_future_cutoff(self):
        actual = self.gpu.batch([0])
        for a, b in zip(actual, self.cpu.batch([0])):
            np.testing.assert_array_equal(a, b)
        self.assertTrue(set(range(350)).issubset(set(actual[1])))
        self.assertTrue(np.all(self.cpu.minute[actual[1]] < 1440))
        self.assertIn(3, actual[6])

    def test_repeated_targets_distinct_minutes_and_buffer_reset(self):
        for ids in ([0, 350, 800, 0, 349, 351], [1500], [1501, 1555, 1501], [1]):
            actual = self.gpu.batch(ids)
            for a, b in zip(actual, self.cpu.batch(ids)):
                np.testing.assert_array_equal(a, b)
            np.testing.assert_array_equal(actual[0][actual[4]], self.cpu.src[ids])
            np.testing.assert_array_equal(actual[0][actual[5]], self.cpu.dst[ids])

    def test_mixed_days_and_invalid_ids_rejected_before_kernel(self):
        for ids in ([0, 1500], [], [-1], [1600]):
            with self.assertRaises(AssertionError):
                self.gpu.batch(ids)
'''


In [ ]:
# tests/test_cuda_features.py
SOURCES['tests/test_cuda_features.py'] = '''import unittest
from types import SimpleNamespace

import numpy as np
import torch

from salt_daily import config
from runtime.graph_runtime import HIInputs
from salt_daily.extra_features import ExtraEdgeFeatures
from salt_daily.cuda_features import CudaFeatureStore

INDICES = [0,1,2,3,4,5,6,7,9,10,12,13,14,15]


def sources():
    rng = np.random.default_rng(712)
    inputs = HIInputs.__new__(HIInputs)
    inputs.edge = rng.normal(size=(24,17)).astype(np.float32)
    inputs.test = None
    for a in (inputs.edge,):
        a[:,12:16] = rng.integers(0,2,(len(a),4))
    inputs.present = np.ones(24,np.uint8); inputs.present[1] = 0
    inputs.test_origin = 10
    inputs.minute = np.r_[np.full(16,7*1440),np.full(8,74*1440)].astype(np.int32)
    inputs.pz = rng.normal(size=24).astype(np.float32)
    inputs.z = rng.normal(size=24).astype(np.float32)
    extra = ExtraEdgeFeatures.__new__(ExtraEdgeFeatures)
    extra.formats = np.resize(np.array([-1,0,6,7,255]),24)
    extra.in_ports = np.arange(24,dtype=np.int32); extra.in_ports[2] = -1
    extra.out_ports = np.arange(24,dtype=np.int32)+16_777_217
    return inputs,extra


@unittest.skipUnless(torch.cuda.is_available(),'CUDA device required')
class CudaFeatureTests(unittest.TestCase):
    def test_exact_fp32_payment_port_all_dates_and_duplicate_order(self):
        inputs,extra = sources()
        store = CudaFeatureStore(inputs,extra,24,INDICES,block=5)
        ids = np.array([22,0,22,3,18,23,4,5])
        expected = extra.append(inputs.edges(ids)[:,INDICES],ids)
        actual = store.edges(torch.tensor(ids,device='cuda'))
        self.assertEqual(actual.dtype,torch.float32)
        np.testing.assert_array_equal(actual.cpu().numpy(),expected)
        self.assertEqual(store.numeric.numel()*4+store.categorical.numel(),24*49)

    def test_missing_base_features_and_ports_fail_on_selection(self):
        inputs,extra = sources()
        store = CudaFeatureStore(inputs,extra,24,INDICES,block=5)
        for tid in (1,2):
            with self.assertRaisesRegex(AssertionError,'Missing'):
                store.edges(torch.tensor([tid],device='cuda'))

    def test_binary_pack_rejects_lossy_input(self):
        inputs,extra = sources(); inputs.edge[0,12] = .5
        with self.assertRaisesRegex(AssertionError,'binary'):
            CudaFeatureStore(inputs,extra,24,INDICES,block=5)

    def test_day_refresh_and_all_eight_inputs_match_cpu(self):
        inputs,extra = sources()
        store = CudaFeatureStore(inputs,extra,24,INDICES,block=5)
        nodes = np.arange(57,dtype=np.float32).reshape(3,19)
        degree = np.arange(6,dtype=np.int64).reshape(3,2)
        graph = [np.array(x,dtype=np.int64) for x in
                 ([0,1,2],[0,17,18],[0,1,2],[1,2,0],[0,0],[1,1],[1,2,3],[1,1,0])]
        ids = np.array([3,3])
        gpu_graph = tuple(torch.tensor(a,device='cuda') for a in graph)
        for shift in (0,100):
            inputs.nodes = nodes+shift
            store.set_day(inputs.nodes,degree+shift)
            expected = list(inputs.materialize(ids,graph))
            expected[2] = extra.append(expected[2][:,INDICES],graph[1])
            expected[4] = extra.append(expected[4][:,INDICES],ids)
            expected.append((degree+shift)[graph[0]].astype(np.float32))
            actual = store.materialize(torch.tensor(ids,device='cuda'),gpu_graph)
            for a,b in zip(actual,expected):
                np.testing.assert_array_equal(a.cpu().numpy(),b)


class EarlySplitTests(unittest.TestCase):
    def test_oversized_graph_splits_before_feature_gather_and_keeps_tail(self):
        from salt_daily.data import DailyData
        from unittest.mock import Mock
        loader = DailyData.__new__(DailyData)
        loader.gpu_features = object()
        loader.topology = SimpleNamespace(batch_tensors=lambda ids: (np.arange(3),np.arange(len(ids)*10)))
        calls = []
        def materialize(ids,graph=None):
            calls.append(list(ids))
            self.assertLessEqual(len(graph[1]),20)
            return ('inputs',),graph
        loader.materialize_gpu = materialize
        data = {'ids':np.arange(10)+100}
        order = np.array([7,2,5,1,8,0,3])
        result = list(loader.microbatches(data,order,microbatch=5,max_edges=20))
        np.testing.assert_array_equal(np.concatenate([r[0] for r in result]),order)
        self.assertTrue(all(len(x)<=2 for x in calls))
        self.assertEqual(sum(map(len,calls)),len(order))
'''


In [ ]:
# tests/test_independent_evaluation.py
SOURCES['tests/test_independent_evaluation.py'] = '''import json
from contextlib import ExitStack
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch

import numpy as np

from test_full_population import dataset, TinyLoader
from salt_daily import training, evaluation
from salt_daily.population import FullPopulation
from salt_daily.storage import read_json, write_json, sha
import run


class IndependentEvaluationTests(unittest.TestCase):
    def setup_run(self, temp):
        root = Path(temp); protocol = dataset(root)
        protocol['variants'] = ['A3','A0']
        protocol['evaluation']['independent_variants'] = ['A3']
        write_json(root/'protocol.json', protocol)
        stack = ExitStack(); self.addCleanup(stack.close)
        for module in (training,run):
            stack.enter_context(patch.object(module,'ROOT',root))
            stack.enter_context(patch.object(module,'RUN',root/'run'))
            stack.enter_context(patch.object(module,'log'))
        stack.enter_context(patch.object(evaluation,'ROOT',root))
        stack.enter_context(patch.object(evaluation,'log'))
        stack.enter_context(patch.object(training,'cohort',lambda scope: FullPopulation(scope,root/'cache/population')))
        stack.enter_context(patch.object(training,'resource_guard'))
        return root,protocol

    def test_individual_a3_full_val_test_without_other_models_and_cached_retry(self):
        with tempfile.TemporaryDirectory() as temp:
            root,p = self.setup_run(temp)
            training.train_variant('A3',p,TinyLoader(),'cpu')
            result = training.evaluate_variant(p,TinyLoader(),'A3','cpu')
            folder = root/'run/models/A3'
            self.assertFalse((root/'run/models/A0').exists())
            self.assertFalse((root/'run/selection_frozen.json').exists())
            self.assertEqual(result['validation']['scored'],4)
            self.assertEqual(result['test']['scored'],4)
            self.assertEqual(result['test']['coverage'],1.)
            self.assertEqual(result['test']['test_authorization']['scope'],'variant')
            before = {str(x):sha(x) for x in folder.rglob('*') if x.is_file() and x.suffix!='jsonl'}
            with patch.object(training,'score',side_effect=AssertionError('rescoring cached results')):
                again = training.evaluate_variant(p,TinyLoader(),'A3','cpu')
            self.assertEqual(again,result)
            self.assertEqual(before,{str(x):sha(x) for x in folder.rglob('*') if x.is_file() and x.suffix!='jsonl'})
            self.assertTrue((folder/'evaluation_complete.json').exists())

    def test_final_test_refuses_incomplete_training_and_missing_freeze(self):
        with tempfile.TemporaryDirectory() as temp:
            root,p = self.setup_run(temp)
            with self.assertRaisesRegex(RuntimeError,'finish all training'):
                training.evaluate_variant(p,TinyLoader(),'A3','cpu')
            with patch.object(run,'verify_ready',return_value=p), patch('salt_daily.data.DailyData') as loader:
                with self.assertRaisesRegex(RuntimeError,'finish all training'):
                    run.evaluate_one('A3','cpu')
                loader.assert_not_called()
            training.train_variant('A3',p,TinyLoader(),'cpu')
            original = training.cohort; opened=[]
            def population(scope):
                opened.append(scope)
                return original(scope)
            with patch.object(training,'cohort',population):
                with self.assertRaisesRegex(RuntimeError,'freeze the checkpoint'):
                    training.evaluate_scope(p,TinyLoader(),'test','cpu',variants=['A3'])
            self.assertNotIn('test',opened)
            self.assertFalse(any((root/'run/models/A3').glob('test_epoch*')))

    def test_cached_test_rejects_changed_validation_and_checkpoint(self):
        with tempfile.TemporaryDirectory() as temp:
            root,p = self.setup_run(temp)
            training.train_variant('A3',p,TinyLoader(),'cpu')
            training.evaluate_variant(p,TinyLoader(),'A3','cpu')
            folder = root/'run/models/A3';path=folder/'validation.json';before=path.read_bytes()
            changed=read_json(path);changed['F1']+=.01;write_json(path,changed)
            with self.assertRaisesRegex(AssertionError,'Frozen variant'):
                training.evaluate_scope(p,TinyLoader(),'test','cpu',variants=['A3'])
            path.write_bytes(before)
            with (folder/'best.pt').open('ab') as handle:handle.write(b'changed')
            with self.assertRaisesRegex(AssertionError,'checkpoint changed'):
                training.evaluate_scope(p,TinyLoader(),'test','cpu',variants=['A3'])

    def test_individual_test_interruption_resumes_only_uncommitted_targets(self):
        class CountingLoader(TinyLoader):
            def __init__(self):self.seen=[]
            def microbatches(self,data,indices,**kwargs):
                if data.scope=='test':self.seen.extend(data['ids'][indices].tolist())
                yield from super().microbatches(data,indices,**kwargs)
        with tempfile.TemporaryDirectory() as temp:
            root,p = self.setup_run(temp)
            training.train_variant('A3',p,TinyLoader(),'cpu')
            loader=CountingLoader();original=training.score
            def interrupted(model,data,loader,device,folder,scope,epoch,stop_check):
                def check():
                    stop_check()
                    if scope=='test' and len(loader.seen)>=2:raise RuntimeError('test interruption')
                return original(model,data,loader,device,folder,scope,epoch,check)
            with patch.object(training,'score',interrupted):
                with self.assertRaisesRegex(RuntimeError,'test interruption'):
                    training.evaluate_variant(p,loader,'A3','cpu')
            self.assertEqual(len(loader.seen),2)
            self.assertFalse((root/'run/models/A3/test.json').exists())
            result=training.evaluate_variant(p,loader,'A3','cpu')
            np.testing.assert_array_equal(loader.seen,FullPopulation('test',root/'cache/population').ids)
            self.assertEqual(result['test']['scored'],4)

    def test_pipeline_finishes_a3_test_before_a0_train_and_reuses_results(self):
        with tempfile.TemporaryDirectory() as temp:
            root,p = self.setup_run(temp)
            train=training.train_variant;score=training.score;calls=[]
            def train_checked(variant,*args,**kwargs):
                if variant=='A0':
                    self.assertTrue((root/'run/models/A3/evaluation_complete.json').exists())
                    self.assertFalse((root/'run/selection_frozen.json').exists())
                return train(variant,*args,**kwargs)
            def score_counted(model,data,loader,device,folder,scope,*args,**kwargs):
                calls.append((model.variant,scope))
                return score(model,data,loader,device,folder,scope,*args,**kwargs)
            with patch.object(training,'train_variant',train_checked),patch.object(training,'score',score_counted):
                run.run_pipeline(p,TinyLoader(),'cpu')
            self.assertEqual(calls.count(('A3','validation')),1)
            self.assertEqual(calls.count(('A3','test')),1)
            self.assertEqual(calls.count(('A0','test')),1)
            result=read_json(root/'run/complete.json')
            expected=max(p['variants'],key=lambda v:result['validation'][v]['F1'])
            self.assertEqual(result['selection']['selected_variant'],expected)
            self.assertEqual(result['test']['A3']['test_authorization']['scope'],'variant')
            self.assertEqual(result['test']['A0']['test_authorization']['scope'],'global')
            self.assertEqual(result['independently_evaluated_variants'],['A3'])
'''


In [ ]:
RUN_UNIT_TESTS = True
if RUN_UNIT_TESTS:
    subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=WORK_DIR,check=True)


In [ ]:
RUN_END_TO_END_SMOKE = True
if RUN_END_TO_END_SMOKE:
    subprocess.run([sys.executable,"-u","-m","salt_daily.smoke"],cwd=WORK_DIR,check=True)


In [ ]:
RUN_CUDA_SMOKE = False  # GPU가 있고 작은 합성 데이터의 CUDA 전체 경로도 확인할 때
if RUN_CUDA_SMOKE:
    subprocess.run([sys.executable,"-u","-m","salt_daily.smoke","--device","cuda"],cwd=WORK_DIR,check=True)


## 검증 범위 · 2026-11-05

단위 테스트에는 CUDA/CPU 그래프 일치, float32 피처·Port·결제형식 일치, 피처 조합 전 분할,
A3 개별 Val/Test, 변경된 checkpoint/Val 거부, 평가 재개·완료 결과 재사용 검사가 포함됩니다.
GPU가 없으면 CUDA 전용 테스트는 skip으로 표시됩니다. 작은 스모크는 CPU에서 원본 전처리부터 전체 순서를 검증합니다.
스모크는 실데이터의 속도·성능·수렴 검증이 아닙니다. 전달본의 실제 검증 기록은 [방법 문서](속도개선_학습방식_논문비교_HGB계획.md)를 확인하세요.
새 전처리와 기존 대규모 캐시의 수치 일치는 별도 검증이 필요합니다. 이전 가중치를 자동 재사용하지 않습니다.
